# Supervised Artifact Classifier — Multi-Anatomy MRI

Three ResNet-18 binary classifiers for MRI motion-artifact detection — supervised baseline for the SSL/unsupervised study (SimCLR, MAE, DINO, DAE, PatchCore).

| # | Anatomy | Clean (label 0) | Artifact (label 1) |
|---|---------|------------------|---------------------|
| 1 | **Knee**     | FastMRI (preprocessed) | KMAR    |
| 2 | **Brain**    | IXI T1+T2 (balanced)   | MR-ART  |
| 3 | **Combined** | FastMRI + IXI          | MR-ART + KMAR |

## Source dataset
Built by `preprocessing-artifact-datasets.ipynb`. Expects the manifest layout:
```
<ARTIFACT_DATA_ROOT>/
├── supervised_clean_pool/{train,val}/{knee,brain}/*.npy
├── supervised_artifacts/{train,val}/{mrart,kmar}/*.npy
└── supervised_manifests/
      ├── supervised_train_{knee,brain,combined}.npz
      └── supervised_val_{knee,brain,combined}.npz
```

## Key design choices (matched to SSL notebooks for fair comparison)
- **Backbone**: ResNet-18 adapted to 1-channel input
- **Loss**: `BCEWithLogitsLoss` with `pos_weight = N_neg / N_pos` (handles ~5.65% positive ratio)
- **20 epochs**, AdamW, cosine schedule with 2-epoch warmup, BS=64 × 4 accum = 256 effective
- **Same augmentations** as SSL training (light: hflip, ±10° rotate, intensity jitter)
- **Best by val AUROC** (not val loss — class-imbalanced)
- **Evaluation**: train + val only (no test sets in this notebook)

In [ ]:
import os, math, time, gc, json, random
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision

from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve,
                              confusion_matrix, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Hyperparameters (mirrors MAE / DINO / SimCLR notebooks)

In [ ]:
BATCH_SIZE     = 64
GRAD_ACCUM     = 4            # effective batch = 256
EPOCHS         = 20
BASE_LR        = 3e-4
WARMUP_EPOCHS  = 2
WEIGHT_DECAY   = 1e-4
BETAS          = (0.9, 0.999)
MAX_GRAD_NORM  = 1.0

IMG_SIZE       = 192
NUM_WORKERS    = 4

TOTAL_TIME_BUDGET = 12 * 3600
SESSION_START = time.time()
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Epochs per model: {EPOCHS}")

## Data — manifest-driven dataset
Each `.npz` manifest stores `paths` and `labels` arrays. Paths were written with prefix `/kaggle/working/artifact_data/` during preprocessing — we rewrite that to the current `ARTIFACT_DATA_ROOT` so the manifests work after the data is uploaded as a Kaggle dataset.

In [ ]:
MANIFEST_WRITE_PREFIX = "/kaggle/working/artifact_data"

def remap_path(p, new_root):
    if p.startswith(MANIFEST_WRITE_PREFIX):
        return new_root.rstrip("/") + p[len(MANIFEST_WRITE_PREFIX):]
    return p

def load_manifest(npz_path, data_root):
    data = np.load(npz_path, allow_pickle=True)
    paths = [remap_path(str(p), data_root) for p in data["paths"]]
    labels = np.asarray(data["labels"]).astype(np.int64)
    missing = [p for p in paths[:20] if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} of first 20 paths missing. First: {missing[0]}\n"
            f"Set ARTIFACT_DATA_ROOT correctly. Manifest prefix expected: {MANIFEST_WRITE_PREFIX}"
        )
    return paths, labels


class ClassifierAugment:
    """Light augmentations -- same recipe as MAE/DINO/SimCLR."""
    def __call__(self, img):
        img = img.copy()
        if random.random() < 0.5:
            img = np.flip(img, axis=1).copy()
        if random.random() < 0.5:
            angle = random.uniform(-10, 10)
            img = scipy.ndimage.rotate(img, angle, reshape=False, order=1, mode='reflect')
        if random.random() < 0.5:
            gain = random.uniform(0.9, 1.1)
            bias = random.uniform(-0.05, 0.05)
            img = img * gain + bias
        img = np.clip(img, 0.0, 1.0).astype(np.float32)
        return torch.from_numpy(img).unsqueeze(0)


class CleanTransform:
    def __call__(self, x):
        return torch.from_numpy(x).unsqueeze(0).float()


class ManifestDataset(Dataset):
    """label 0=clean, 1=artifact."""
    def __init__(self, paths, labels, transform=None, name=""):
        self.paths = list(paths)
        self.labels = np.asarray(labels).astype(np.int64)
        self.transform = transform or CleanTransform()
        n_pos = int((self.labels == 1).sum()); n_neg = int((self.labels == 0).sum())
        ratio = 100.0 * n_pos / max(1, n_pos + n_neg)
        print(f"  [{name}] {len(self.paths)} slices | clean={n_neg} | artifact={n_pos} ({ratio:.2f}%)")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)
        return self.transform(img), int(self.labels[idx])

## Model — ResNet-18 (1-channel) with binary head

In [ ]:
def build_resnet18_binary(pretrained=False):
    weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    m = torchvision.models.resnet18(weights=weights)
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    nn.init.kaiming_normal_(m.conv1.weight, mode='fan_out', nonlinearity='relu')
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m

def count_params(model):
    return sum(p.numel() for p in model.parameters())

_m = build_resnet18_binary()
print(f"ResNet-18 params: {count_params(_m):,} ({count_params(_m)/1e6:.1f}M)")
del _m

## Training & evaluation infrastructure

In [ ]:
def log_time_budget(phase=""):
    elapsed = time.time() - SESSION_START
    remaining = TOTAL_TIME_BUDGET - elapsed
    print(f"\u23f1 [{phase}] Elapsed: {elapsed/3600:.2f}h | Remaining: {remaining/3600:.2f}h")
    return remaining

def log_gpu(prefix=""):
    if device == "cuda":
        a = torch.cuda.memory_allocated()/1e9
        p = torch.cuda.max_memory_allocated()/1e9
        t = torch.cuda.get_device_properties(0).total_memory/1e9
        print(f"  \U0001f5a5 GPU [{prefix}]: {a:.2f}/{t:.1f} GB (peak {p:.2f} GB)")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    log_gpu("after clear")
    log_time_budget("GPU cleared")


def train_epoch(model, loader, optimizer, scaler, loss_fn):
    model.train()
    total = 0.0; n_batches = len(loader)
    optimizer.zero_grad()
    log_int = max(1, n_batches // 5)
    epoch_start = time.time()
    for step, (imgs, y) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        y = y.float().to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(imgs).squeeze(1)
            loss = loss_fn(logits, y) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == n_batches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total += loss.item() * GRAD_ACCUM
        if (step + 1) % log_int == 0:
            avg = total / (step + 1)
            rate = (step + 1) / max(1e-6, time.time() - epoch_start)
            eta = (n_batches - step - 1) / rate
            print(f"    batch {step+1}/{n_batches} | loss={avg:.4f} | {rate:.1f} batch/s | ETA {eta:.0f}s", end="\r")
    print(" " * 80, end="\r")
    return total / n_batches


@torch.no_grad()
def predict(model, loader, desc=""):
    model.eval()
    probs, labels = [], []
    it = loader if not desc else tqdm(loader, desc=desc)
    for imgs, y in it:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(imgs).squeeze(1).float()
        probs.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)


def binary_metrics(probs, labels, threshold=0.5):
    out = {}
    if len(np.unique(labels)) >= 2:
        out["auroc"] = float(roc_auc_score(labels, probs))
        out["auprc"] = float(average_precision_score(labels, probs))
    else:
        out["auroc"] = float("nan"); out["auprc"] = float("nan")
    pred = (probs >= threshold).astype(int)
    out["acc"] = float((pred == labels).mean())
    out["f1"] = float(f1_score(labels, pred, zero_division=0))
    out["flag_rate"] = float(pred.mean())
    out["threshold"] = float(threshold)
    return out

In [ ]:
def train_classifier(train_loader, val_loader, save_dir, model_name="sup",
                     pos_weight=None, pretrained=False):
    os.makedirs(save_dir, exist_ok=True)
    log_time_budget(f"START {model_name}")

    print(f"\n{'='*60}\n  \U0001f680 Training: {model_name}\n  \U0001f4c1 {save_dir}\n{'='*60}")
    print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")
    print(f"  pos_weight={None if pos_weight is None else f'{pos_weight:.2f}'}, EPOCHS={EPOCHS}, LR={BASE_LR}")

    model = build_resnet18_binary(pretrained=pretrained).to(device)
    print(f"  Parameters: {count_params(model):,}"); log_gpu("model loaded")

    if pos_weight is not None:
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR,
                                  weight_decay=WEIGHT_DECAY, betas=BETAS)

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    history = {"train_loss": [], "val_loss": [], "val_auroc": [], "val_auprc": [],
                "val_f1": [], "lr": [], "epoch_time": []}
    best_auroc = -1.0
    total_start = time.time()

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        t_loss = train_epoch(model, train_loader, optimizer, scaler, loss_fn)

        # Val loss + metrics
        model.eval()
        v_total = 0.0
        with torch.no_grad():
            for imgs, y in val_loader:
                imgs = imgs.to(device); y = y.float().to(device)
                with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                    logits = model(imgs).squeeze(1)
                    v_total += loss_fn(logits, y).item()
        v_loss = v_total / max(1, len(val_loader))
        v_probs, v_labels = predict(model, val_loader)
        v_metrics = binary_metrics(v_probs, v_labels, 0.5)

        scheduler.step()
        epoch_time = time.time() - epoch_start
        lr_now = scheduler.get_last_lr()[0]
        history["train_loss"].append(t_loss); history["val_loss"].append(v_loss)
        history["val_auroc"].append(v_metrics["auroc"]); history["val_auprc"].append(v_metrics["auprc"])
        history["val_f1"].append(v_metrics["f1"]); history["lr"].append(lr_now)
        history["epoch_time"].append(epoch_time)

        elapsed_model = time.time() - total_start
        rem = TOTAL_TIME_BUDGET - (time.time() - SESSION_START)
        print(f"  [{model_name}] Ep {epoch+1:2d}/{EPOCHS} | "
              f"train={t_loss:.4f} val={v_loss:.4f} | AUROC={v_metrics['auroc']:.4f} "
              f"AUPRC={v_metrics['auprc']:.4f} F1={v_metrics['f1']:.3f} | "
              f"lr={lr_now:.5f} | {epoch_time:.0f}s/ep | model {elapsed_model/60:.1f}m | "
              f"left {rem/3600:.1f}h")

        if v_metrics["auroc"] > best_auroc and not math.isnan(v_metrics["auroc"]):
            best_auroc = v_metrics["auroc"]
            torch.save({"epoch": epoch+1, "model": model.state_dict(),
                        "val_auroc": best_auroc, "val_metrics": v_metrics},
                       os.path.join(save_dir, "best.pt"))
            print(f"       -> New best AUROC={best_auroc:.4f}")

        if (epoch + 1) % 5 == 0:
            torch.save({"epoch": epoch+1, "model": model.state_dict()},
                       os.path.join(save_dir, f"epoch_{epoch+1}.pt"))

        if rem < 1800 and epoch < EPOCHS - 1:
            print(f"  WARNING TIME: {rem/60:.0f}m left, stopping at ep {epoch+1}"); break

    torch.save({"epoch": epoch+1, "model": model.state_dict(),
                "history": history}, os.path.join(save_dir, "final.pt"))
    print(f"\n  Done {model_name} | best val AUROC = {best_auroc:.4f}")
    log_time_budget(f"END {model_name}")

    best = torch.load(os.path.join(save_dir, "best.pt"), map_location=device)
    model.load_state_dict(best["model"])
    return model, history

In [ ]:
def run_analysis(model, train_clean_loader, val_loader, history, save_dir, model_name):
    """Train + val analysis only (no test sets)."""
    print(f"\n{'='*60}\n  Analysis: {model_name}\n{'='*60}")
    log_time_budget(f"ANALYSIS START {model_name}")

    # -- 1. Curves --
    print("[1/6] Training curves...")
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    fig.suptitle(f"{model_name} -- Training Curves", fontsize=14, y=1.02)
    axes[0].plot(history['train_loss'], label='Train', lw=2)
    axes[0].plot(history['val_loss'], label='Val', lw=2)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['val_auroc'], lw=2, color='tab:purple')
    axes[1].set_title('Val AUROC'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0.5, 1.0)
    axes[2].plot(history['val_auprc'], lw=2, color='tab:cyan')
    axes[2].set_title('Val AUPRC'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)
    axes[3].plot(history['lr'], lw=2, color='tab:green')
    axes[3].set_title('LR'); axes[3].set_xlabel('Epoch'); axes[3].grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=150, bbox_inches='tight'); plt.show()

    # -- 2. Predict train + val --
    print("[2/6] Predicting on train + val...")
    train_probs, train_labels = predict(model, train_clean_loader, desc="train")
    val_probs, val_labels     = predict(model, val_loader,         desc="val")
    train_metrics = binary_metrics(train_probs, train_labels, 0.5)
    val_metrics   = binary_metrics(val_probs,   val_labels,   0.5)
    print(f"  Train: AUROC={train_metrics['auroc']:.4f} AUPRC={train_metrics['auprc']:.4f} "
          f"F1={train_metrics['f1']:.3f} acc={train_metrics['acc']:.3f}")
    print(f"  Val:   AUROC={val_metrics['auroc']:.4f} AUPRC={val_metrics['auprc']:.4f} "
          f"F1={val_metrics['f1']:.3f} acc={val_metrics['acc']:.3f}")

    # -- 3. Score histograms --
    print("[3/6] Score distributions...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    fig.suptitle(f"{model_name} -- Score Distributions", fontsize=14, y=1.02)
    axes[0].hist(train_probs[train_labels == 0], bins=60, alpha=0.6, color='steelblue', label='Clean (0)', density=True)
    axes[0].hist(train_probs[train_labels == 1], bins=60, alpha=0.6, color='coral', label='Artifact (1)', density=True)
    axes[0].axvline(0.5, color='gray', ls='--', alpha=0.7, label='thresh=0.5')
    axes[0].set_xlabel('P(artifact)'); axes[0].set_title('Train'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].hist(val_probs[val_labels == 0], bins=60, alpha=0.6, color='steelblue', label='Clean (0)', density=True)
    axes[1].hist(val_probs[val_labels == 1], bins=60, alpha=0.6, color='coral', label='Artifact (1)', density=True)
    axes[1].axvline(0.5, color='gray', ls='--', alpha=0.7)
    axes[1].set_xlabel('P(artifact)'); axes[1].set_title('Val'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(save_dir, 'score_dist.png'), dpi=150, bbox_inches='tight'); plt.show()

    # -- 4. ROC + PR curves (val) --
    print("[4/6] ROC + PR curves...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{model_name} -- Val ROC & PR", fontsize=14, y=1.02)
    if len(np.unique(val_labels)) >= 2:
        fpr, tpr, _ = roc_curve(val_labels, val_probs)
        auroc = roc_auc_score(val_labels, val_probs)
        axes[0].plot(fpr, tpr, lw=2, label=f'AUROC={auroc:.4f}')
        axes[0].plot([0, 1], [0, 1], color='gray', ls='--', alpha=0.5)
        axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC')
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        prec, rec, _ = precision_recall_curve(val_labels, val_probs)
        auprc = average_precision_score(val_labels, val_probs)
        axes[1].plot(rec, prec, lw=2, color='coral', label=f'AUPRC={auprc:.4f}')
        axes[1].axhline(val_labels.mean(), color='gray', ls='--', alpha=0.5,
                        label=f'baseline={val_labels.mean():.3f}')
        axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(save_dir, 'roc_pr.png'), dpi=150, bbox_inches='tight'); plt.show()

    # -- 5. Confusion matrix (val @ 0.5) --
    print("[5/6] Confusion matrix @ 0.5...")
    pred = (val_probs >= 0.5).astype(int)
    cm = confusion_matrix(val_labels, pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm.max()/2 else 'black', fontsize=12)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Clean', 'Artifact']); ax.set_yticklabels(['Clean', 'Artifact'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{model_name} -- Val Confusion (thr=0.5)')
    plt.colorbar(im, ax=ax, shrink=0.7)
    fig.tight_layout(); fig.savefig(os.path.join(save_dir, 'confusion.png'), dpi=150, bbox_inches='tight'); plt.show()

    # -- 6. Threshold sweep (percentiles of train clean scores) --
    print("[6/6] Threshold sweep (train clean -> percentile thresholds)...")
    train_probs_neg = train_probs[train_labels == 0]
    thresholds = {}
    for pct in [90, 95, 97.5, 99]:
        thr = float(np.percentile(train_probs_neg, pct))
        flagged = float((val_probs >= thr).mean())
        flag_pos = float((val_probs[val_labels == 1] >= thr).mean()) if (val_labels == 1).any() else float('nan')
        flag_neg = float((val_probs[val_labels == 0] >= thr).mean()) if (val_labels == 0).any() else float('nan')
        thresholds[f"p{pct}"] = thr
        print(f"    p{pct}: thr={thr:.4f} | val flagged={flagged:.3f} | "
              f"recall(pos)={flag_pos:.3f} | FPR(neg)={flag_neg:.3f}")

    # -- Save predictions --
    np.savez(os.path.join(save_dir, 'predictions.npz'),
             train_probs=train_probs, train_labels=train_labels,
             val_probs=val_probs,     val_labels=val_labels,
             train_metrics=json.dumps(train_metrics),
             val_metrics=json.dumps(val_metrics),
             thresholds=json.dumps(thresholds))

    log_time_budget(f"ANALYSIS END {model_name}")
    return train_probs, train_labels, val_probs, val_labels, val_metrics

## Configure paths
Set `ARTIFACT_DATA_ROOT` to the folder that contains `supervised_manifests/`, `supervised_clean_pool/`, `supervised_artifacts/`.

In [ ]:
# UPDATE THIS to point at the uploaded artifact_data root
ARTIFACT_DATA_ROOT = "/kaggle/input/datasets/kaustubhratna/mrartifact"

MANIFEST_DIR = os.path.join(ARTIFACT_DATA_ROOT, "supervised_manifests")

for p in [ARTIFACT_DATA_ROOT, MANIFEST_DIR]:
    print(f"  [{'OK' if os.path.exists(p) else 'MISS'}] {p}")

if os.path.exists(MANIFEST_DIR):
    print("\nAvailable manifests:")
    for f in sorted(os.listdir(MANIFEST_DIR)):
        print(f"  - {f}")

In [ ]:
def build_train_val_loaders(variant):
    """variant in {'knee', 'brain', 'combined'}.
    Returns (train_loader, val_loader, train_clean_loader, pos_weight)."""
    tr_npz = os.path.join(MANIFEST_DIR, f"supervised_train_{variant}.npz")
    va_npz = os.path.join(MANIFEST_DIR, f"supervised_val_{variant}.npz")
    tr_paths, tr_labels = load_manifest(tr_npz, ARTIFACT_DATA_ROOT)
    va_paths, va_labels = load_manifest(va_npz, ARTIFACT_DATA_ROOT)

    aug = ClassifierAugment()
    train_ds = ManifestDataset(tr_paths, tr_labels, transform=aug, name=f"{variant}-train")
    val_ds   = ManifestDataset(va_paths, va_labels, transform=CleanTransform(), name=f"{variant}-val")
    train_clean = ManifestDataset(tr_paths, tr_labels, transform=CleanTransform(), name=f"{variant}-train-clean")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    train_clean_loader = DataLoader(train_clean, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

    n_pos = int((np.asarray(tr_labels) == 1).sum())
    n_neg = int((np.asarray(tr_labels) == 0).sum())
    pos_weight = n_neg / max(1, n_pos)
    print(f"  pos_weight = N_neg/N_pos = {n_neg}/{n_pos} = {pos_weight:.2f}")
    return train_loader, val_loader, train_clean_loader, pos_weight

---
# Part 1 — Knee only
Clean: FastMRI knee | Artifact: KMAR

In [ ]:
knee_train_loader, knee_val_loader, knee_train_clean_loader, knee_pos_w = \
    build_train_val_loaders("knee")

In [ ]:
KNEE_SAVE = "/kaggle/working/checkpoints/sup_knee"
knee_model, knee_history = train_classifier(
    knee_train_loader, knee_val_loader, KNEE_SAVE,
    model_name="Sup-Knee", pos_weight=knee_pos_w)

In [ ]:
knee_results = run_analysis(
    knee_model, knee_train_clean_loader, knee_val_loader,
    knee_history, KNEE_SAVE, "Sup-Knee")

In [ ]:
del knee_model, knee_train_loader, knee_val_loader, knee_train_clean_loader
clear_gpu()

---
# Part 2 — Brain only
Clean: IXI T1+T2 (balanced) | Artifact: MR-ART

In [ ]:
brain_train_loader, brain_val_loader, brain_train_clean_loader, brain_pos_w = \
    build_train_val_loaders("brain")

In [ ]:
BRAIN_SAVE = "/kaggle/working/checkpoints/sup_brain"
brain_model, brain_history = train_classifier(
    brain_train_loader, brain_val_loader, BRAIN_SAVE,
    model_name="Sup-Brain", pos_weight=brain_pos_w)

In [ ]:
brain_results = run_analysis(
    brain_model, brain_train_clean_loader, brain_val_loader,
    brain_history, BRAIN_SAVE, "Sup-Brain")

In [ ]:
del brain_model, brain_train_loader, brain_val_loader, brain_train_clean_loader
clear_gpu()

---
# Part 3 — Combined
Clean: FastMRI + IXI | Artifact: MR-ART + KMAR

In [ ]:
comb_train_loader, comb_val_loader, comb_train_clean_loader, comb_pos_w = \
    build_train_val_loaders("combined")

In [ ]:
COMB_SAVE = "/kaggle/working/checkpoints/sup_combined"
comb_model, comb_history = train_classifier(
    comb_train_loader, comb_val_loader, COMB_SAVE,
    model_name="Sup-Combined", pos_weight=comb_pos_w)

In [ ]:
comb_results = run_analysis(
    comb_model, comb_train_clean_loader, comb_val_loader,
    comb_history, COMB_SAVE, "Sup-Combined")

In [ ]:
del comb_model, comb_train_loader, comb_val_loader, comb_train_clean_loader
clear_gpu()

---
# Cross-model summary

In [ ]:
print("\n" + "=" * 80)
print("  SUPERVISED CLASSIFIER -- CROSS-MODEL SUMMARY (val only)")
print("=" * 80)

histories = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMB_SAVE)]:
    histories[name] = torch.load(os.path.join(save_dir, "final.pt"), map_location="cpu")["history"]

print(f"\n{'Model':<12} {'BestEp':<8} {'Val AUROC':<11} {'Val AUPRC':<11} {'Val F1':<8} {'AvgEp(s)':<10}")
print("-" * 65)
for name, h in histories.items():
    valid = [v for v in h["val_auroc"] if not math.isnan(v)]
    if not valid:
        continue
    best_ep = int(np.nanargmax(h["val_auroc"])) + 1
    best_auroc = max(valid)
    best_auprc = h["val_auprc"][best_ep - 1]
    best_f1 = h["val_f1"][best_ep - 1]
    avg_t = float(np.mean(h["epoch_time"]))
    print(f"{name:<12} {best_ep:<8} {best_auroc:<11.4f} {best_auprc:<11.4f} {best_f1:<8.3f} {avg_t:<10.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Supervised -- Model Comparison", fontsize=14, y=1.02)
colors = {"Knee": "tab:blue", "Brain": "tab:orange", "Combined": "tab:green"}
for name, h in histories.items():
    axes[0].plot(h['val_loss'], label=name, lw=2, color=colors[name])
axes[0].set_title('Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
for name, h in histories.items():
    axes[1].plot(h['val_auroc'], label=name, lw=2, color=colors[name])
axes[1].set_title('Val AUROC'); axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0.5, 1.0)
axes[1].legend(); axes[1].grid(True, alpha=0.3)
for name, h in histories.items():
    axes[2].plot(h['val_auprc'], label=name, lw=2, color=colors[name])
axes[2].set_title('Val AUPRC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig('sup_comparison_plots.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# import shutil
# from IPython.display import FileLink, display
# for name, save_dir in [("knee", KNEE_SAVE), ("brain", BRAIN_SAVE), ("combined", COMB_SAVE)]:
#     zp = shutil.make_archive(f"sup_{name}_checkpoints", "zip", root_dir=".", base_dir=save_dir)
#     print(f"  {name}: {zp} ({os.path.getsize(zp)/1e6:.1f} MB)"); display(FileLink(zp))
# all_zip = shutil.make_archive("sup_all_checkpoints", "zip", root_dir=".", base_dir="checkpoints")
# print(f"\n  ALL: {all_zip} ({os.path.getsize(all_zip)/1e6:.1f} MB)"); display(FileLink(all_zip))